# AnaliseSiplanRPS — staging → wh_siplan_rps

Lê a tabela de staging criada pelo **Dataflow Gen2** `df_analisesiplan_rps`
e grava na tabela `wh_siplan_rps.dbo.analise_siplan_rps`.

| Etapa | Responsável |
|-------|-------------|
| SharePoint → lake_prep_siplan | Dataflow Gen2 `df_analisesiplan_rps` |
| lake_prep_siplan → wh_siplan_rps | **Este notebook** |
| Edições por linha | Power Automate (SQL connector no warehouse) |

> **Recarga completa**: `mode=overwrite` recria a tabela a cada execução.
> Edições do Power Automate em colunas extras do warehouse serão perdidas
> na próxima recarga — adapte para MERGE incremental quando necessário.

In [ ]:
# ── Parâmetros ──────────────────────────────────────────────────────────────
STAGING_TABLE = 'lake_prep_siplan.dbo.analisesiplan_rps_raw'

SQL_ENDPOINT  = (
    'beu5bmmdbuwedpv62ucm524jzi-dmrv7k3fbwbevh5d4sidg3urfq'
    '.datawarehouse.fabric.microsoft.com'
)
WAREHOUSE_DB  = 'wh_siplan_rps'
TARGET_TABLE  = 'analise_siplan_rps'

In [ ]:
# ── Imports ──────────────────────────────────────────────────────────────────
from datetime import datetime

try:
    from notebookutils import mssparkutils
    HAS_MSSPARKUTILS = True
except ImportError:
    HAS_MSSPARKUTILS = False

print(f'Início: {datetime.now():%d/%m/%Y %H:%M}')

In [ ]:
# ── Leitura da tabela de staging (populada pelo Dataflow Gen2) ───────────────
df_spark = spark.sql(f'SELECT * FROM {STAGING_TABLE}')

total = df_spark.count()
print(f'{total} registros em {STAGING_TABLE}')
print('Schema:')
df_spark.printSchema()
df_spark.show(5, truncate=80)

In [ ]:
# ── Escrita no Fabric Warehouse via Spark JDBC ───────────────────────────────
def get_db_token() -> str:
    if HAS_MSSPARKUTILS:
        return mssparkutils.credentials.getToken('https://database.windows.net/.default')
    from azure.identity import InteractiveBrowserCredential, DeviceCodeCredential
    try:
        cred = InteractiveBrowserCredential()
    except Exception:
        cred = DeviceCodeCredential()
    return cred.get_token('https://database.windows.net/.default').token


jdbc_url = (
    f'jdbc:sqlserver://{SQL_ENDPOINT}'
    f';database={WAREHOUSE_DB}'
    f';encrypt=true;trustServerCertificate=false'
)

print(f'Gravando em {WAREHOUSE_DB}.dbo.{TARGET_TABLE}...')

(
    df_spark.write
    .format('jdbc')
    .option('url', jdbc_url)
    .option('dbtable', f'dbo.{TARGET_TABLE}')
    .option('accessToken', get_db_token())
    .option('driver', 'com.microsoft.sqlserver.jdbc.SQLServerDriver')
    .option('batchsize', 1000)
    .mode('overwrite')
    .save()
)

print(f'OK — dbo.{TARGET_TABLE} criada/atualizada em {WAREHOUSE_DB}.')

In [ ]:
# ── Verificação ───────────────────────────────────────────────────────────────
count = (
    spark.read
    .format('jdbc')
    .option('url', jdbc_url)
    .option('dbtable', f'dbo.{TARGET_TABLE}')
    .option('accessToken', get_db_token())
    .option('driver', 'com.microsoft.sqlserver.jdbc.SQLServerDriver')
    .load()
    .count()
)
print(f'Verificação: {count} registros em {WAREHOUSE_DB}.dbo.{TARGET_TABLE}')

if HAS_MSSPARKUTILS:
    mssparkutils.notebook.exit(str(count))